In [1]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import Dataset, DataLoader
import cv2
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2



In [2]:
DATA_ROOT = "/kaggle/input/cassava-leaf-disease-classification"
test_image_dir = f"{DATA_ROOT}/test_images"
train_image_dir = f"{DATA_ROOT}/train_images"
sample_sub_path = f"{DATA_ROOT}/sample_submission.csv"
train_csv_path = f"{DATA_ROOT}/train.csv"



In [3]:
test_df = pd.read_csv(sample_sub_path)
train_df = pd.read_csv(train_csv_path)
test_df.head()



,image_id,label
0,1234294272.jpg,4
1,1234332763.jpg,4
2,1234375577.jpg,4
3,1234555380.jpg,4
4,1234571117.jpg,4


In [4]:
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



In [5]:
RESNET_MEAN = (0.485, 0.456, 0.406)
RESNET_STD = (0.229, 0.224, 0.225)

_eff_w = models.EfficientNet_V2_S_Weights.DEFAULT
_eff_t = _eff_w.transforms()
EFFNET_MEAN = tuple(float(x) for x in _eff_t.mean)
EFFNET_STD = tuple(float(x) for x in _eff_t.std)

_CLAHE_P = 0.35
_CLAHE_CLIP = 1.5

# Change (score-relevant, minimal): make inference deterministic by disabling CLAHE randomness at test time.
# Prototype extraction keeps the existing CLAHE settings to preserve your established prototype construction logic.
resnet_transforms = A.Compose(
    [
        A.CLAHE(clip_limit=_CLAHE_CLIP, tile_grid_size=(8, 8), p=0.0),
        A.Resize(224, 224),
        A.Normalize(mean=RESNET_MEAN, std=RESNET_STD),
        ToTensorV2(),
    ]
)
efficientnet_transforms = A.Compose(
    [
        A.CLAHE(clip_limit=_CLAHE_CLIP, tile_grid_size=(8, 8), p=0.0),
        A.Resize(384, 384),
        A.Normalize(mean=EFFNET_MEAN, std=EFFNET_STD),
        ToTensorV2(),
    ]
)

resnet_transforms_proto = A.Compose(
    [
        A.CLAHE(clip_limit=_CLAHE_CLIP, tile_grid_size=(8, 8), p=_CLAHE_P),
        A.Resize(224, 224),
        A.Normalize(mean=RESNET_MEAN, std=RESNET_STD),
        ToTensorV2(),
    ]
)
efficientnet_transforms_proto = A.Compose(
    [
        A.CLAHE(clip_limit=_CLAHE_CLIP, tile_grid_size=(8, 8), p=_CLAHE_P),
        A.Resize(384, 384),
        A.Normalize(mean=EFFNET_MEAN, std=EFFNET_STD),
        ToTensorV2(),
    ]
)




In [6]:
class CassavaTestDataset(Dataset):
    def __init__(
        self, dataframe, image_dir, transform_resnet=None, transform_efficientnet=None
    ):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform_resnet = transform_resnet
        self.transform_efficientnet = transform_efficientnet

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx, 0]  # image_id
        img_path = os.path.join(self.image_dir, img_name)

        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform_resnet:
            augmented_resnet = self.transform_resnet(image=image)
            image_resnet = augmented_resnet["image"]
        else:
            image_resnet = None

        if self.transform_efficientnet:
            augmented_efficientnet = self.transform_efficientnet(image=image)
            image_efficientnet = augmented_efficientnet["image"]
        else:
            image_efficientnet = None

        return image_resnet, image_efficientnet, img_name




In [7]:
test_dataset = CassavaTestDataset(
    test_df,
    test_image_dir,
    transform_resnet=resnet_transforms,
    transform_efficientnet=efficientnet_transforms,
)

_num_workers = 2 if os.cpu_count() and os.cpu_count() > 2 else 0

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=_num_workers,
    pin_memory=torch.cuda.is_available(),
)



In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device




device(type='cuda')

In [9]:
def find_weight_file(preferred_path: str, filename_fallback: str):
    p = Path(preferred_path)
    if p.exists():
        return str(p)
    matches = list(Path("/kaggle/input").rglob(filename_fallback))
    if matches:
        return str(sorted(matches)[0])
    return None


def _clean_state_dict(state):
    if (
        isinstance(state, dict)
        and "state_dict" in state
        and isinstance(state["state_dict"], dict)
    ):
        state = state["state_dict"]
    if not isinstance(state, dict):
        return state
    if any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}
    return state


resnet_weights_path = find_weight_file(
    "/kaggle/input/casava-aug/pytorch/default/1/cassava_leaf_best_model_fine_aug.pth",
    "cassava_leaf_best_model_fine_aug.pth",
)

if resnet_weights_path is None:
    resnet_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    resnet_model.fc = nn.Identity()
    print(
        "ResNet external weights not found; using torchvision ImageNet weights + prototype mapping."
    )
else:
    resnet_model = models.resnet50(weights=None)
    state = torch.load(resnet_weights_path, map_location="cpu")
    state = _clean_state_dict(state)
    resnet_model.load_state_dict(state, strict=True)
    print(f"Loaded ResNet weights from: {resnet_weights_path}")

resnet_model = resnet_model.to(device)
resnet_model.eval()



Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:01<00:00, 83.8MB/s]


ResNet external weights not found; using torchvision ImageNet weights + prototype mapping.


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [10]:
eff_weights_path = find_weight_file(
    "/kaggle/input/eff-t/pytorch/default/1/Eff.pth", "Eff.pth"
)

if eff_weights_path is None:
    efficientnet_model = models.efficientnet_v2_s(
        weights=models.EfficientNet_V2_S_Weights.DEFAULT
    )
    efficientnet_model.classifier[1] = nn.Identity()
    print(
        "EfficientNet external weights not found; using torchvision ImageNet weights + prototype mapping."
    )
else:
    efficientnet_model = models.efficientnet_v2_s(weights=None)
    state_eff = torch.load(eff_weights_path, map_location="cpu")
    state_eff = _clean_state_dict(state_eff)
    efficientnet_model.load_state_dict(state_eff, strict=True)
    print(f"Loaded EfficientNet weights from: {eff_weights_path}")

efficientnet_model = efficientnet_model.to(device)
efficientnet_model.eval()




Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth
100%|██████████| 82.7M/82.7M [00:00<00:00, 99.1MB/s]


EfficientNet external weights not found; using torchvision ImageNet weights + prototype mapping.


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  

In [11]:
class CassavaTrainProtoDataset(Dataset):
    def __init__(
        self, dataframe, image_dir, transform_resnet=None, transform_efficientnet=None
    ):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform_resnet = transform_resnet
        self.transform_efficientnet = transform_efficientnet

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, "image_id"]
        label = int(self.df.loc[idx, "label"])
        img_path = os.path.join(self.image_dir, img_name)

        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        augmented_resnet = self.transform_resnet(image=image)
        augmented_efficientnet = self.transform_efficientnet(image=image)

        return augmented_resnet["image"], augmented_efficientnet["image"], label


def _is_identity_head_resnet(m):
    return isinstance(m.fc, nn.Identity)


def _is_identity_head_eff(m):
    return (
        hasattr(m, "classifier")
        and isinstance(m.classifier, nn.Sequential)
        and len(m.classifier) > 1
        and isinstance(m.classifier[1], nn.Identity)
    )


use_prototypes = _is_identity_head_resnet(resnet_model) and _is_identity_head_eff(
    efficientnet_model
)
print("Using prototype mapping:", use_prototypes)

rng = np.random.RandomState(SEED)
per_class = 1400  # keep as-is for stability/time

proto_rows = []
labels_np = train_df["label"].values
for c in range(5):
    idxs = np.where(labels_np == c)[0]
    perm = rng.permutation(len(idxs))
    idxs = idxs[perm]
    take = min(per_class, len(idxs))
    chosen = idxs[:take]
    proto_rows.append(train_df.iloc[chosen])
proto_df = pd.concat(proto_rows, axis=0).reset_index(drop=True)

proto_dataset = CassavaTrainProtoDataset(
    proto_df,
    train_image_dir,
    transform_resnet=resnet_transforms_proto,
    transform_efficientnet=efficientnet_transforms_proto,
)
proto_loader = DataLoader(
    proto_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=_num_workers,
    pin_memory=torch.cuda.is_available(),
)

if use_prototypes:
    resnet_model.eval()
    efficientnet_model.eval()

    with torch.no_grad():
        img_r0, img_e0, _ = next(iter(proto_loader))
        img_r0 = img_r0.to(device, non_blocking=True)
        img_e0 = img_e0.to(device, non_blocking=True)
        d_r = int(resnet_model(img_r0).shape[1])
        d_e = int(efficientnet_model(img_e0).shape[1])

    sums_r = torch.zeros(5, d_r, device=device, dtype=torch.float32)
    sums_e = torch.zeros(5, d_e, device=device, dtype=torch.float32)
    counts = torch.zeros(5, device=device, dtype=torch.float32)

    with torch.no_grad():
        for img_r, img_e, y in proto_loader:
            img_r = img_r.to(device, non_blocking=True)
            img_e = img_e.to(device, non_blocking=True)
            y = torch.as_tensor(y, device=device, dtype=torch.long)

            feat_r = resnet_model(img_r).to(torch.float32)
            feat_e = efficientnet_model(img_e).to(torch.float32)

            feat_r = F.normalize(feat_r, p=2, dim=1)
            feat_e = F.normalize(feat_e, p=2, dim=1)

            sums_r.index_add_(0, y, feat_r)
            sums_e.index_add_(0, y, feat_e)
            counts.index_add_(0, y, torch.ones_like(y, dtype=torch.float32))

    proto_r = sums_r / counts.unsqueeze(1).clamp_min(1.0)  # [5, d_r]
    proto_e = sums_e / counts.unsqueeze(1).clamp_min(1.0)  # [5, d_e]

    proto_r = F.normalize(proto_r.to(torch.float32), p=2, dim=1)
    proto_e = F.normalize(proto_e.to(torch.float32), p=2, dim=1)
else:
    proto_r, proto_e = None, None



Using prototype mapping: True


In [12]:
ensemble_predictions = []
image_names = []

if use_prototypes:
    proto_r_dev = proto_r
    proto_e_dev = proto_e
else:
    proto_r_dev, proto_e_dev = None, None

COSINE_TEMPERATURE = 5.0

SIM_W_EFF = 0.60
SIM_W_RES = 1.0 - SIM_W_EFF

with torch.no_grad():
    for images_resnet, images_efficientnet, img_names in test_loader:
        images_resnet = images_resnet.to(device, non_blocking=True)
        images_efficientnet = images_efficientnet.to(device, non_blocking=True)

        outputs_resnet = resnet_model(images_resnet).to(torch.float32)
        outputs_efficientnet = efficientnet_model(images_efficientnet).to(torch.float32)

        if use_prototypes:
            feat_r = F.normalize(outputs_resnet, p=2, dim=1)
            feat_e = F.normalize(outputs_efficientnet, p=2, dim=1)

            sims_r = feat_r @ proto_r_dev.t()  # [B,5]
            sims_e = feat_e @ proto_e_dev.t()  # [B,5]

            probs_r = F.softmax(sims_r * COSINE_TEMPERATURE, dim=1)
            probs_e = F.softmax(sims_e * COSINE_TEMPERATURE, dim=1)
            probs = SIM_W_RES * probs_r + SIM_W_EFF * probs_e

            preds = probs.argmax(dim=1).detach().cpu().numpy()
        else:
            probs_resnet = F.softmax(outputs_resnet, dim=1)
            probs_efficientnet = F.softmax(outputs_efficientnet, dim=1)
            combined_probs = (probs_resnet + probs_efficientnet) / 2.0
            preds = combined_probs.argmax(dim=1).detach().cpu().numpy()

        ensemble_predictions.extend(preds.tolist())
        image_names.extend(list(img_names))

len(image_names), len(ensemble_predictions)



(2676, 2676)

In [13]:
if len(image_names) != len(test_df):
    raise RuntimeError(
        f"Prediction count mismatch: got {len(image_names)} preds for {len(test_df)} test rows."
    )

if image_names != test_df["image_id"].tolist():
    pred_map = dict(zip(image_names, ensemble_predictions))
    ensemble_predictions = [pred_map[i] for i in test_df["image_id"].tolist()]
    image_names = test_df["image_id"].tolist()

submission_df = pd.DataFrame(
    {"image_id": image_names, "label": np.asarray(ensemble_predictions, dtype=int)}
)

assert (
    submission_df.shape[0] == test_df.shape[0]
), "Row count mismatch vs sample_submission."
assert list(submission_df.columns) == [
    "image_id",
    "label",
], "Submission columns must be: image_id,label"
assert not submission_df["label"].isna().any(), "Found NaN labels in submission."
assert submission_df["label"].between(0, 4).all(), "Labels must be in {0,1,2,3,4}."

submission_df.to_csv("submission.csv", index=False)
print("Submission file saved as 'submission.csv'")
submission_df.head()

Submission file saved as 'submission.csv'


,image_id,label
0,1234294272.jpg,3
1,1234332763.jpg,1
2,1234375577.jpg,1
3,1234555380.jpg,4
4,1234571117.jpg,3
